In [5]:
import os
os.environ["FINNHUB_API_KEY"] = "d8plsf9r01qujv4iu09gd8plsf9r01qujv4iu0a0"
import json
import time
from pathlib import Path

import requests
import pandas as pd
from sqlalchemy import text

%run db_config.ipynb

FINNHUB_API_KEY = os.environ.get("FINNHUB_API_KEY")
if not FINNHUB_API_KEY:
    raise ValueError("Set FINNHUB_API_KEY environment variable before running")
PROFILE_URL = "https://finnhub.io/api/v1/stock/profile2"
REQUEST_TIMEOUT = 30
THROTTLE_SECONDS = 1.2  # 60 req/min free tier
CHECKPOINT_FILE = Path("python/checkpoint/profile_checkpoint.json")

def load_symbol_id_map(conn) -> dict[str, int]:
    rows = conn.execute(text("SELECT id, symbol FROM stocks ORDER BY symbol")).fetchall()
    return {row.symbol: row.id for row in rows}

def profile_exists(conn, stock_id: int) -> bool:
    row = conn.execute(
        text("SELECT 1 FROM stock_profiles WHERE stock_id = :stock_id"),
        {"stock_id": stock_id},
    ).fetchone()
    return row is not None

def fetch_profile(symbol: str) -> dict | None:
    response = requests.get(
        PROFILE_URL,
        params={"symbol": symbol, "token": FINNHUB_API_KEY},
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    data = response.json()
    if not data or "name" not in data:
        return None
    return data

def map_profile(data: dict, stock_id: int) -> dict:
    return {
        "stock_id": stock_id,
        "industry": data.get("finnhubIndustry") or "Unknown",
        "market_cap": data.get("marketCapitalization"),
        "logo": data.get("logo"),
        "company_name": data.get("name"),
        "shares": data.get("shareOutstanding"),
    }

def insert_profile(conn, profile: dict) -> None:
    conn.execute(
        text(
            "INSERT INTO stock_profiles "
            "(industry, market_cap, logo, company_name, shares, stock_id) "
            "VALUES (:industry, :market_cap, :logo, :company_name, :shares, :stock_id)"
        ),
        profile,
    )

def load_checkpoint() -> set[str]:
    if CHECKPOINT_FILE.exists():
        return set(json.loads(CHECKPOINT_FILE.read_text()))
    return set()

def save_checkpoint(done: set[str]) -> None:
    CHECKPOINT_FILE.parent.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_FILE.write_text(json.dumps(sorted(done)))

def main():
    engine = get_engine()
    done = load_checkpoint()
    inserted = 0
    skipped = 0
    failed = 0
    no_data = 0

    with engine.begin() as conn:
        symbol_map = load_symbol_id_map(conn)

    for symbol, stock_id in symbol_map.items():
        if symbol in done:
            print(f"SKIP {symbol} (checkpoint)")
            skipped += 1
            continue

        try:
            with engine.begin() as conn:
                if profile_exists(conn, stock_id):
                    print(f"SKIP {symbol} (already in DB)")
                    done.add(symbol)
                    save_checkpoint(done)
                    skipped += 1
                    continue
            data = fetch_profile(symbol)
            if data is None:
                print(f"WARN {symbol}: no profile data from Finnhub")
                no_data += 1
                done.add(symbol)
                save_checkpoint(done)
                time.sleep(THROTTLE_SECONDS)
                continue

            profile = map_profile(data, stock_id)

            # Required NOT NULL fields guard
            if profile["market_cap"] is None or profile["shares"] is None:
                print(f"WARN {symbol}: missing market_cap or shares")
                no_data += 1
                done.add(symbol)
                save_checkpoint(done)
                time.sleep(THROTTLE_SECONDS)
                continue

            with engine.begin() as conn:
                insert_profile(conn, profile)

            print(f"OK {symbol}: profile inserted")
            inserted += 1
            done.add(symbol)
            save_checkpoint(done)

        except requests.HTTPError as e:
            print(f"FAIL {symbol}: HTTP {e.response.status_code}")
            failed += 1
        except Exception as e:
            print(f"FAIL {symbol}: {e}")
            failed += 1

        time.sleep(THROTTLE_SECONDS)

    print(f"\nDone. inserted={inserted}, skipped={skipped}, no_data={no_data}, failed={failed}")
    print(f"Completed {len(done)} / {len(symbol_map)} symbols")

main()


SKIP A (checkpoint)
SKIP AAPL (checkpoint)
SKIP ABBV (checkpoint)
SKIP ABNB (checkpoint)
SKIP ABT (checkpoint)
SKIP ACGL (checkpoint)
SKIP ACN (checkpoint)
SKIP ADBE (checkpoint)
SKIP ADI (checkpoint)
SKIP ADM (checkpoint)
SKIP ADP (checkpoint)
SKIP ADSK (checkpoint)
SKIP AEE (checkpoint)
SKIP AEP (checkpoint)
SKIP AES (checkpoint)
SKIP AFL (checkpoint)
SKIP AIG (checkpoint)
SKIP AIZ (checkpoint)
SKIP AJG (checkpoint)
SKIP AKAM (checkpoint)
SKIP ALB (checkpoint)
SKIP ALGN (checkpoint)
SKIP ALL (checkpoint)
SKIP ALLE (checkpoint)
SKIP AMAT (checkpoint)
SKIP AMCR (checkpoint)
SKIP AMD (checkpoint)
SKIP AME (checkpoint)
SKIP AMGN (checkpoint)
SKIP AMP (checkpoint)
SKIP AMT (checkpoint)
SKIP AMZN (checkpoint)
SKIP ANET (checkpoint)
SKIP AON (checkpoint)
SKIP AOS (checkpoint)
SKIP APA (checkpoint)
SKIP APD (checkpoint)
SKIP APH (checkpoint)
SKIP APO (checkpoint)
SKIP APP (checkpoint)
SKIP APTV (checkpoint)
SKIP ARE (checkpoint)
SKIP ARES (checkpoint)
SKIP ATO (checkpoint)
SKIP AVB (checkpoi